# Advanced Python Custom Sequences — Problems with Solutions

This notebook contains advanced practice problems on custom sequence types in Python.

Topics covered:

- `__getitem__`
- integer indexing
- negative indexing
- slicing with `slice` objects
- `slice.indices()`
- iteration through `__getitem__`
- `IndexError` and sequence termination
- `__len__`
- lazy sequence design
- custom Fibonacci-style sequences
- read-only sequence behavior
- practical sequence validation

## Problem 1 — Inspect What `__getitem__` Receives

Create a class named `InspectorSequence` whose `__getitem__` method returns information about the key it received.

Requirements:

1. If the key is an integer, return `("int", key)`.
2. If the key is a slice, return `("slice", start, stop, step)`.
3. Test it with:
   - `seq[3]`
   - `seq[-1]`
   - `seq[1:5]`
   - `seq[::2]`
   - `seq[::-1]`

The goal is to see that Python passes an `int` for normal indexing and a `slice` object for slicing.

In [1]:
# Solution 1

class InspectorSequence:
    def __getitem__(self, key):
        if isinstance(key, int):
            return ("int", key)
        elif isinstance(key, slice):
            return ("slice", key.start, key.stop, key.step)
        else:
            raise TypeError("indices must be integers or slices")


seq = InspectorSequence()

results = [
    seq[3],
    seq[-1],
    seq[1:5],
    seq[::2],
    seq[::-1],
]

results

[('int', 3),
 ('int', -1),
 ('slice', 1, 5, None),
 ('slice', None, None, 2),
 ('slice', None, None, -1)]

### Explanation

Python translates bracket syntax into calls to `__getitem__`.

For example:

```python
seq[3]
```

calls:

```python
seq.__getitem__(3)
```

But:

```python
seq[1:5:2]
```

calls something conceptually equivalent to:

```python
seq.__getitem__(slice(1, 5, 2))
```

## Problem 2 — Build a Minimal Read-Only Sequence

Create a class named `ReadOnlySequence` that wraps an internal list.

Requirements:

1. Support integer indexing.
2. Support negative indexing.
3. Raise `IndexError` for out-of-bounds indexes.
4. Support iteration using Python's default sequence iteration protocol.
5. Do not implement `__iter__` yet.

Hint: if a class implements `__getitem__`, Python can iterate over it by calling index `0`, then `1`, then `2`, and so on until `IndexError` is raised.

In [2]:
# Solution 2

class ReadOnlySequence:
    def __init__(self, data):
        self._data = list(data)

    def __getitem__(self, index):
        if not isinstance(index, int):
            raise TypeError("index must be an integer")

        if index < 0:
            index = len(self._data) + index

        if index < 0 or index >= len(self._data):
            raise IndexError("index out of range")

        return self._data[index]


# Checks
seq = ReadOnlySequence([10, 20, 30, 40])

assert seq[0] == 10
assert seq[3] == 40
assert seq[-1] == 40
assert seq[-4] == 10
assert list(seq) == [10, 20, 30, 40]

try:
    seq[4]
except IndexError as ex:
    positive_error = str(ex)

try:
    seq[-5]
except IndexError as ex:
    negative_error = str(ex)

positive_error, negative_error, list(seq)

('index out of range', 'index out of range', [10, 20, 30, 40])

### Explanation

A class can be iterable even without `__iter__` if it supports integer indexing through `__getitem__` and raises `IndexError` when the iteration should stop.

This is why correct bounds handling matters. If `IndexError` is never raised, iteration may become infinite.

## Problem 3 — Add Slice Support to the Read-Only Sequence

Extend the previous idea by creating `SliceableReadOnlySequence`.

Requirements:

1. Support integer indexing.
2. Support negative indexing.
3. Support slicing.
4. A slice should return a new `SliceableReadOnlySequence`, not a raw list.
5. Implement `__repr__` so results are easy to inspect.
6. Implement `to_list()` for testing.

In [3]:
# Solution 3

class SliceableReadOnlySequence:
    def __init__(self, data):
        self._data = list(data)

    def __getitem__(self, key):
        if isinstance(key, int):
            if key < 0:
                key = len(self._data) + key

            if key < 0 or key >= len(self._data):
                raise IndexError("index out of range")

            return self._data[key]

        if isinstance(key, slice):
            return type(self)(self._data[key])

        raise TypeError("indices must be integers or slices")

    def __repr__(self):
        return f"{type(self).__name__}({self._data!r})"

    def to_list(self):
        return self._data.copy()


# Checks
seq = SliceableReadOnlySequence([0, 1, 2, 3, 4, 5])

assert seq[0] == 0
assert seq[-1] == 5
assert seq[1:5].to_list() == [1, 2, 3, 4]
assert seq[::2].to_list() == [0, 2, 4]
assert seq[::-1].to_list() == [5, 4, 3, 2, 1, 0]
assert isinstance(seq[1:3], SliceableReadOnlySequence)

seq[::-2]

SliceableReadOnlySequence([5, 3, 1])

### Explanation

When `key` is a slice, the wrapped list already knows how to perform slicing correctly.

The important design choice is what to return. Returning `type(self)(...)` means slicing the custom sequence returns another custom sequence.

## Problem 4 — Show the Normalized Slice Range

Create a function `slice_report(s, length)` that returns a dictionary containing:

1. The original slice parts: `start`, `stop`, `step`.
2. The normalized tuple from `s.indices(length)`.
3. The exact list of visited indices.

Test it with tricky slices, especially negative-step slices.

In [4]:
# Solution 4

def slice_report(s, length):
    normalized = s.indices(length)
    visited = list(range(*normalized))
    return {
        "original": {
            "start": s.start,
            "stop": s.stop,
            "step": s.step,
        },
        "normalized": normalized,
        "visited": visited,
    }


reports = [
    slice_report(slice(None, None, None), 6),
    slice_report(slice(None, None, -1), 6),
    slice_report(slice(1, 100, 2), 6),
    slice_report(slice(3, -1, -1), 6),
    slice_report(slice(3, None, -1), 6),
]

reports

[{'original': {'start': None, 'stop': None, 'step': None},
  'normalized': (0, 6, 1),
  'visited': [0, 1, 2, 3, 4, 5]},
 {'original': {'start': None, 'stop': None, 'step': -1},
  'normalized': (5, -1, -1),
  'visited': [5, 4, 3, 2, 1, 0]},
 {'original': {'start': 1, 'stop': 100, 'step': 2},
  'normalized': (1, 6, 2),
  'visited': [1, 3, 5]},
 {'original': {'start': 3, 'stop': -1, 'step': -1},
  'normalized': (3, 5, -1),
  'visited': []},
 {'original': {'start': 3, 'stop': None, 'step': -1},
  'normalized': (3, -1, -1),
  'visited': [3, 2, 1, 0]}]

### Explanation

`slice.indices(length)` converts a slice into a concrete `(start, stop, step)` triple for a sequence of a specific length.

Then:

```python
range(*s.indices(length))
```

gives the exact indices that the slice visits.

## Problem 5 — A Debuggable Sequence

Create a class named `DebugSequence` that wraps a list and records every access.

Requirements:

1. Integer indexing should return one item.
2. Slicing should return a list of selected values.
3. Every access should be appended to `history`.
4. For integer access, store the original index and the normalized index.
5. For slice access, store the original slice, the normalized tuple, and the visited indices.
6. Raise appropriate errors for bad indexes.

In [5]:
# Solution 5

class DebugSequence:
    def __init__(self, data):
        self._data = list(data)
        self.history = []

    def __getitem__(self, key):
        if isinstance(key, int):
            original = key
            if key < 0:
                key = len(self._data) + key

            if key < 0 or key >= len(self._data):
                raise IndexError("index out of range")

            self.history.append({
                "type": "index",
                "original": original,
                "normalized": key,
            })
            return self._data[key]

        if isinstance(key, slice):
            normalized = key.indices(len(self._data))
            visited = list(range(*normalized))
            self.history.append({
                "type": "slice",
                "original": key,
                "normalized": normalized,
                "visited": visited,
            })
            return [self._data[i] for i in visited]

        raise TypeError("indices must be integers or slices")


# Checks
seq = DebugSequence("abcdef")

assert seq[0] == "a"
assert seq[-1] == "f"
assert seq[1:5:2] == ["b", "d"]
assert seq[::-2] == ["f", "d", "b"]

seq.history

[{'type': 'index', 'original': 0, 'normalized': 0},
 {'type': 'index', 'original': -1, 'normalized': 5},
 {'type': 'slice',
  'original': slice(1, 5, 2),
  'normalized': (1, 5, 2),
  'visited': [1, 3]},
 {'type': 'slice',
  'original': slice(None, None, -2),
  'normalized': (5, -1, -2),
  'visited': [5, 3, 1]}]

### Explanation

This problem combines integer normalization and slice normalization.

For integer indexing, negative indexes are adjusted manually.

For slicing, `slice.indices()` performs the normalization for us.

## Problem 6 — Implement `__len__` and Explain Why It Matters

Create a class named `SizedSequence` that supports:

1. `len(seq)`
2. integer indexing
3. slicing
4. iteration
5. membership checks using `in`

Use a wrapped list internally.

In [6]:
# Solution 6

class SizedSequence:
    def __init__(self, data):
        self._data = list(data)

    def __len__(self):
        return len(self._data)

    def __getitem__(self, key):
        if isinstance(key, int):
            if key < 0:
                key = len(self) + key
            if key < 0 or key >= len(self):
                raise IndexError("index out of range")
            return self._data[key]

        if isinstance(key, slice):
            return type(self)(self._data[key])

        raise TypeError("indices must be integers or slices")

    def __repr__(self):
        return f"{type(self).__name__}({self._data!r})"

    def to_list(self):
        return self._data.copy()


# Checks
seq = SizedSequence([10, 20, 30, 40])

assert len(seq) == 4
assert seq[-1] == 40
assert list(seq) == [10, 20, 30, 40]
assert 30 in seq
assert 99 not in seq
assert seq[1:].to_list() == [20, 30, 40]

len(seq), list(seq), 30 in seq, seq[1:]

(4, [10, 20, 30, 40], True, SizedSequence([20, 30, 40]))

### Explanation

`__len__` allows Python's built-in `len()` function to work.

Together, `__getitem__` and `__len__` make an object behave much more like a real sequence.

## Problem 7 — Build a Bounded Fibonacci Sequence

Create a class named `FibSequence` representing the first `n` Fibonacci numbers.

Use this Fibonacci definition:

```python
fib(0) = 1
fib(1) = 1
fib(2) = 2
fib(3) = 3
```

Requirements:

1. The sequence should be bounded by `n`.
2. Support `len(f)`.
3. Support positive integer indexing.
4. Support negative integer indexing.
5. Raise `IndexError` when out of bounds.
6. Support slicing.
7. Slicing should return a list of Fibonacci numbers.
8. Use caching so repeated Fibonacci calculations are efficient.

In [7]:
# Solution 7

from functools import lru_cache


class FibSequence:
    def __init__(self, n):
        if n < 0:
            raise ValueError("n must be non-negative")
        self._n = n

    def __len__(self):
        return self._n

    def __getitem__(self, key):
        if isinstance(key, int):
            if key < 0:
                key = len(self) + key

            if key < 0 or key >= len(self):
                raise IndexError("index out of range")

            return self._fib(key)

        if isinstance(key, slice):
            start, stop, step = key.indices(len(self))
            return [self._fib(i) for i in range(start, stop, step)]

        raise TypeError("indices must be integers or slices")

    @staticmethod
    @lru_cache(maxsize=None)
    def _fib(index):
        if index < 2:
            return 1
        return FibSequence._fib(index - 1) + FibSequence._fib(index - 2)


# Checks
f = FibSequence(10)

assert len(f) == 10
assert f[0] == 1
assert f[1] == 1
assert f[2] == 2
assert f[3] == 3
assert f[-1] == 55
assert f[:5] == [1, 1, 2, 3, 5]
assert f[5::-1] == [8, 5, 3, 2, 1, 1]
assert list(f) == [1, 1, 2, 3, 5, 8, 13, 21, 34, 55]

list(f), f[::-1]

([1, 1, 2, 3, 5, 8, 13, 21, 34, 55], [55, 34, 21, 13, 8, 5, 3, 2, 1, 1])

### Explanation

The class is bounded because it has a fixed length `n`.

That means `f[10]` should raise `IndexError` when `len(f) == 10`.

Slicing is handled by normalizing the slice into a range of valid indices:

```python
start, stop, step = key.indices(len(self))
```

Then each selected index is converted into its Fibonacci value.

## Problem 8 — Prevent Infinite Iteration

A broken custom Fibonacci class returns a value for every integer index, even indexes outside its intended length.

Explain why this is dangerous, then fix the class.

Broken version:

```python
class BrokenFib:
    def __init__(self, n):
        self._n = n

    def __getitem__(self, index):
        return index
```

Requirements for the fixed version:

1. Implement `__len__`.
2. Raise `IndexError` when the index is outside `0 <= index < n`.
3. Support negative indexes correctly.
4. Demonstrate that `list(fixed)` terminates.

In [8]:
# Solution 8

class FixedFibLike:
    def __init__(self, n):
        if n < 0:
            raise ValueError("n must be non-negative")
        self._n = n

    def __len__(self):
        return self._n

    def __getitem__(self, index):
        if not isinstance(index, int):
            raise TypeError("index must be an integer")

        if index < 0:
            index = len(self) + index

        if index < 0 or index >= len(self):
            raise IndexError("index out of range")

        # Placeholder value for demonstration.
        return index


# Checks
fixed = FixedFibLike(5)

assert list(fixed) == [0, 1, 2, 3, 4]
assert fixed[-1] == 4

try:
    fixed[5]
except IndexError as ex:
    error = str(ex)

list(fixed), error

([0, 1, 2, 3, 4], 'index out of range')

### Explanation

When Python iterates over an object using `__getitem__`, it starts at index `0`, then tries `1`, then `2`, and so on.

The iteration stops only when `IndexError` is raised.

If a custom sequence never raises `IndexError`, then iteration may continue forever.

## Problem 9 — Create a Lazy Arithmetic Progression Sequence

Create a class named `ArithmeticProgression` that represents a bounded arithmetic sequence without storing all values.

For example:

```python
ArithmeticProgression(start=10, step=3, length=5)
```

represents:

```python
[10, 13, 16, 19, 22]
```

Requirements:

1. Do not pre-generate the full sequence.
2. Support `len()`.
3. Support integer indexing.
4. Support negative indexing.
5. Support slicing.
6. Slicing should return a list.
7. Support iteration.

In [9]:
# Solution 9

class ArithmeticProgression:
    def __init__(self, start, step, length):
        if length < 0:
            raise ValueError("length must be non-negative")
        self.start = start
        self.step = step
        self._length = length

    def __len__(self):
        return self._length

    def _value_at(self, index):
        return self.start + index * self.step

    def __getitem__(self, key):
        if isinstance(key, int):
            if key < 0:
                key = len(self) + key

            if key < 0 or key >= len(self):
                raise IndexError("index out of range")

            return self._value_at(key)

        if isinstance(key, slice):
            start, stop, step = key.indices(len(self))
            return [self._value_at(i) for i in range(start, stop, step)]

        raise TypeError("indices must be integers or slices")

    def __repr__(self):
        return (
            f"ArithmeticProgression(start={self.start!r}, "
            f"step={self.step!r}, length={self._length!r})"
        )


# Checks
ap = ArithmeticProgression(start=10, step=3, length=5)

assert len(ap) == 5
assert ap[0] == 10
assert ap[1] == 13
assert ap[-1] == 22
assert ap[:] == [10, 13, 16, 19, 22]
assert ap[::-1] == [22, 19, 16, 13, 10]
assert list(ap) == [10, 13, 16, 19, 22]

ap, ap[::2], ap[::-2]

(ArithmeticProgression(start=10, step=3, length=5), [10, 16, 22], [22, 16, 10])

### Explanation

This is a lazy sequence because it stores only the formula parameters, not every value.

The value at index `i` is computed only when requested:

```python
start + i * step
```

## Problem 10 — Return a Same-Type Slice for a Lazy Sequence

Improve the arithmetic progression idea so slicing returns another `ArithmeticProgressionView` instead of a list whenever possible.

Example:

```python
seq = ArithmeticProgressionView(10, 3, 8)
seq[1:6:2]
```

should represent values at original indices `1, 3, 5`, which are:

```python
[13, 19, 25]
```

Requirements:

1. Support `len()`.
2. Support integer indexing.
3. Support slicing.
4. A slice should return another `ArithmeticProgressionView`.
5. Provide `to_list()` for inspection.

In [10]:
# Solution 10

class ArithmeticProgressionView:
    def __init__(self, start, step, length):
        if length < 0:
            raise ValueError("length must be non-negative")
        self.start = start
        self.step = step
        self._length = length

    def __len__(self):
        return self._length

    def _value_at(self, index):
        return self.start + index * self.step

    def __getitem__(self, key):
        if isinstance(key, int):
            if key < 0:
                key = len(self) + key

            if key < 0 or key >= len(self):
                raise IndexError("index out of range")

            return self._value_at(key)

        if isinstance(key, slice):
            normalized_start, normalized_stop, normalized_step = key.indices(len(self))
            selected_indices = range(normalized_start, normalized_stop, normalized_step)
            new_length = len(selected_indices)

            if new_length == 0:
                return type(self)(self.start, self.step, 0)

            new_start = self._value_at(normalized_start)
            new_step = self.step * normalized_step
            return type(self)(new_start, new_step, new_length)

        raise TypeError("indices must be integers or slices")

    def to_list(self):
        return [self[i] for i in range(len(self))]

    def __repr__(self):
        return (
            f"ArithmeticProgressionView(start={self.start!r}, "
            f"step={self.step!r}, length={self._length!r})"
        )


# Checks
seq = ArithmeticProgressionView(10, 3, 8)

assert seq.to_list() == [10, 13, 16, 19, 22, 25, 28, 31]

sliced = seq[1:6:2]
assert isinstance(sliced, ArithmeticProgressionView)
assert sliced.to_list() == [13, 19, 25]

reversed_seq = seq[::-1]
assert reversed_seq.to_list() == [31, 28, 25, 22, 19, 16, 13, 10]

sliced, sliced.to_list(), reversed_seq

(ArithmeticProgressionView(start=13, step=6, length=3),
 [13, 19, 25],
 ArithmeticProgressionView(start=31, step=-3, length=8))

### Explanation

A slice of an arithmetic progression is still an arithmetic progression.

If the original sequence has step `d`, and the slice uses step `k`, then the new sequence has step:

```python
d * k
```

This allows slicing to remain lazy.

## Problem 11 — Validate a Custom Sequence Against a Python List

Write a helper function `compare_to_list(custom_seq, reference)` that verifies whether a custom sequence behaves like a list for common operations.

Requirements:

Check that all of the following match the reference list:

1. `len(custom_seq)`
2. `list(custom_seq)`
3. positive indexing
4. negative indexing
5. several slices
6. out-of-bounds positive indexing raises `IndexError`
7. out-of-bounds negative indexing raises `IndexError`

Return a dictionary of test results.

In [11]:
# Solution 11

def raises_index_error(func):
    try:
        func()
    except IndexError:
        return True
    except Exception:
        return False
    return False


def compare_to_list(custom_seq, reference):
    tests = {}

    tests["length"] = len(custom_seq) == len(reference)
    tests["iteration"] = list(custom_seq) == reference

    tests["positive_indexing"] = all(
        custom_seq[i] == reference[i]
        for i in range(len(reference))
    )

    tests["negative_indexing"] = all(
        custom_seq[-i] == reference[-i]
        for i in range(1, len(reference) + 1)
    ) if reference else True

    slices = [
        slice(None, None, None),
        slice(1, None, None),
        slice(None, -1, None),
        slice(None, None, 2),
        slice(None, None, -1),
        slice(3, -1, -1),
    ]

    tests["slicing"] = all(
        list(custom_seq[s]) == list(reference[s])
        for s in slices
    )

    tests["positive_oob_index_error"] = raises_index_error(
        lambda: custom_seq[len(reference)]
    )

    tests["negative_oob_index_error"] = raises_index_error(
        lambda: custom_seq[-len(reference) - 1]
    )

    tests["all_passed"] = all(tests.values())
    return tests


# Use SizedSequence from Problem 6
reference = [10, 20, 30, 40, 50]
custom = SizedSequence(reference)

compare_to_list(custom, reference)

{'length': True,
 'iteration': True,
 'positive_indexing': True,
 'negative_indexing': True,
 'slicing': True,
 'positive_oob_index_error': True,
 'negative_oob_index_error': True,
 'all_passed': True}

### Explanation

When building custom sequence types, it is useful to test them against Python's built-in list behavior.

This kind of test catches common mistakes in:

- negative indexing,
- slicing,
- out-of-bounds handling,
- iteration termination.

## Problem 12 — Create a Read-Only Sequence of Squares

Create a class named `Squares` representing the bounded sequence:

```python
0^2, 1^2, 2^2, ..., (n-1)^2
```

Requirements:

1. Store only `n`, not the full list of squares.
2. Support `len()`.
3. Support positive indexing.
4. Support negative indexing.
5. Support slicing.
6. Support iteration.
7. Slicing should return a list of square values.
8. Raise `IndexError` correctly.

In [12]:
# Solution 12

class Squares:
    def __init__(self, n):
        if n < 0:
            raise ValueError("n must be non-negative")
        self._n = n

    def __len__(self):
        return self._n

    def _value_at(self, index):
        return index ** 2

    def __getitem__(self, key):
        if isinstance(key, int):
            if key < 0:
                key = len(self) + key

            if key < 0 or key >= len(self):
                raise IndexError("index out of range")

            return self._value_at(key)

        if isinstance(key, slice):
            start, stop, step = key.indices(len(self))
            return [self._value_at(i) for i in range(start, stop, step)]

        raise TypeError("indices must be integers or slices")


# Checks
sq = Squares(8)

assert len(sq) == 8
assert sq[0] == 0
assert sq[3] == 9
assert sq[-1] == 49
assert sq[:] == [0, 1, 4, 9, 16, 25, 36, 49]
assert sq[::-1] == [49, 36, 25, 16, 9, 4, 1, 0]
assert list(sq) == [0, 1, 4, 9, 16, 25, 36, 49]

compare_to_list(sq, [0, 1, 4, 9, 16, 25, 36, 49])

{'length': True,
 'iteration': True,
 'positive_indexing': True,
 'negative_indexing': True,
 'slicing': True,
 'positive_oob_index_error': True,
 'negative_oob_index_error': True,
 'all_passed': True}

### Explanation

`Squares` is another lazy bounded sequence.

The class stores only the length. Values are computed when requested.

## Problem 13 — A Sequence That Rejects Slice Assignment

Create a class named `ImmutableSequence` that behaves like a read-only sequence.

Requirements:

1. Support `len()`.
2. Support indexing.
3. Support slicing.
4. Do not implement `__setitem__`.
5. Demonstrate that item assignment fails.
6. Demonstrate that slice assignment fails.

This reinforces the idea that implementing `__getitem__` supports reading, but not mutation.

In [13]:
# Solution 13

class ImmutableSequence:
    def __init__(self, data):
        self._data = tuple(data)

    def __len__(self):
        return len(self._data)

    def __getitem__(self, key):
        if isinstance(key, int):
            return self._data[key]

        if isinstance(key, slice):
            return type(self)(self._data[key])

        raise TypeError("indices must be integers or slices")

    def __repr__(self):
        return f"{type(self).__name__}({list(self._data)!r})"

    def to_list(self):
        return list(self._data)


seq = ImmutableSequence([1, 2, 3, 4])

assert len(seq) == 4
assert seq[0] == 1
assert seq[-1] == 4
assert seq[1:3].to_list() == [2, 3]

try:
    seq[0] = 99
except TypeError as ex:
    item_assignment_error = str(ex)

try:
    seq[1:3] = [99, 100]
except TypeError as ex:
    slice_assignment_error = str(ex)

item_assignment_error, slice_assignment_error

("'ImmutableSequence' object does not support item assignment",
 "'ImmutableSequence' object does not support item assignment")

### Explanation

`__getitem__` supports read access.

To support item assignment, a class must implement `__setitem__`.

Since `ImmutableSequence` does not implement `__setitem__`, both item assignment and slice assignment fail.

## Problem 14 — Build a Sequence of Rows with Column Slicing

Create a class named `TableColumnSequence` that wraps rows of data and allows column extraction using slicing-like behavior.

Input:

```python
rows = [
    ("Ada", "Lovelace", 36),
    ("Grace", "Hopper", 85),
    ("Alan", "Turing", 41),
]
```

Requirements:

1. `table[i]` should return row `i`.
2. `table[i:j]` should return a new `TableColumnSequence` containing those rows.
3. `table.column(k)` should return all values from column `k`.
4. `table.columns(slice_obj)` should return selected columns from every row.
5. Support `len()` and iteration.

In [14]:
# Solution 14

class TableColumnSequence:
    def __init__(self, rows):
        self._rows = [tuple(row) for row in rows]

    def __len__(self):
        return len(self._rows)

    def __getitem__(self, key):
        if isinstance(key, int):
            return self._rows[key]

        if isinstance(key, slice):
            return type(self)(self._rows[key])

        raise TypeError("indices must be integers or slices")

    def column(self, index):
        return [row[index] for row in self._rows]

    def columns(self, s):
        if not isinstance(s, slice):
            raise TypeError("columns expects a slice object")
        return [row[s] for row in self._rows]

    def to_list(self):
        return self._rows.copy()

    def __repr__(self):
        return f"{type(self).__name__}({self._rows!r})"


# Checks
rows = [
    ("Ada", "Lovelace", 36),
    ("Grace", "Hopper", 85),
    ("Alan", "Turing", 41),
]

table = TableColumnSequence(rows)

assert len(table) == 3
assert table[0] == ("Ada", "Lovelace", 36)
assert table[1:].to_list() == [("Grace", "Hopper", 85), ("Alan", "Turing", 41)]
assert table.column(0) == ["Ada", "Grace", "Alan"]
assert table.column(-1) == [36, 85, 41]
assert table.columns(slice(0, 2)) == [("Ada", "Lovelace"), ("Grace", "Hopper"), ("Alan", "Turing")]

table.column(0), table.columns(slice(0, 2)), list(table)

(['Ada', 'Grace', 'Alan'],
 [('Ada', 'Lovelace'), ('Grace', 'Hopper'), ('Alan', 'Turing')],
 [('Ada', 'Lovelace', 36), ('Grace', 'Hopper', 85), ('Alan', 'Turing', 41)])

### Explanation

This problem shows how sequence ideas can be combined with structured data.

The table itself is a sequence of rows.

Each row is also a sequence, so normal indexing and slicing can be applied inside each row.

## Problem 15 — Final Challenge: A Production-Style Bounded Lazy Sequence Base Class

Create an abstract-style base class named `LazySequence`.

It should handle all common sequence mechanics:

1. Store the sequence length.
2. Implement `__len__`.
3. Implement integer indexing.
4. Implement negative indexing.
5. Implement slicing.
6. Raise `IndexError` correctly.
7. Delegate actual value computation to a method named `_value_at(index)`.

Then create two subclasses:

1. `LazySquares`
2. `LazyCubes`

Slicing should return a list of computed values.

In [15]:
# Solution 15

class LazySequence:
    def __init__(self, length):
        if length < 0:
            raise ValueError("length must be non-negative")
        self._length = length

    def __len__(self):
        return self._length

    def _normalize_index(self, index):
        if index < 0:
            index = len(self) + index

        if index < 0 or index >= len(self):
            raise IndexError("index out of range")

        return index

    def __getitem__(self, key):
        if isinstance(key, int):
            index = self._normalize_index(key)
            return self._value_at(index)

        if isinstance(key, slice):
            start, stop, step = key.indices(len(self))
            return [self._value_at(i) for i in range(start, stop, step)]

        raise TypeError("indices must be integers or slices")

    def _value_at(self, index):
        raise NotImplementedError("subclasses must implement _value_at")


class LazySquares(LazySequence):
    def _value_at(self, index):
        return index ** 2


class LazyCubes(LazySequence):
    def _value_at(self, index):
        return index ** 3


# Checks
squares = LazySquares(6)
cubes = LazyCubes(6)

assert len(squares) == 6
assert list(squares) == [0, 1, 4, 9, 16, 25]
assert squares[-1] == 25
assert squares[::-1] == [25, 16, 9, 4, 1, 0]

assert len(cubes) == 6
assert list(cubes) == [0, 1, 8, 27, 64, 125]
assert cubes[-1] == 125
assert cubes[::2] == [0, 8, 64]

compare_to_list(squares, [0, 1, 4, 9, 16, 25]), compare_to_list(cubes, [0, 1, 8, 27, 64, 125])

({'length': True,
  'iteration': True,
  'positive_indexing': True,
  'negative_indexing': True,
  'slicing': True,
  'positive_oob_index_error': True,
  'negative_oob_index_error': True,
  'all_passed': True},
 {'length': True,
  'iteration': True,
  'positive_indexing': True,
  'negative_indexing': True,
  'slicing': True,
  'positive_oob_index_error': True,
  'negative_oob_index_error': True,
  'all_passed': True})

### Explanation

`LazySequence` centralizes the reusable mechanics of bounded sequence behavior.

Subclasses only need to define how a value is computed at a valid index.

This avoids duplicating the same indexing, slicing, negative-index handling, and bounds-checking logic in every custom sequence class.

# Summary

In this notebook, you practiced advanced custom sequence design in Python.

Key ideas:

- `obj[i]` calls `obj.__getitem__(i)`.
- `obj[start:stop:step]` passes a `slice` object to `__getitem__`.
- A class can support iteration by implementing `__getitem__` and raising `IndexError` at the correct time.
- `__len__` enables `len(obj)` and helps make custom objects feel sequence-like.
- Negative integer indexing must usually be handled manually.
- Slice normalization should be handled using `slice.indices(length)`.
- Lazy sequences compute values on demand instead of storing everything.
- A well-designed base class can reuse sequence mechanics across many custom sequence types.